# 🛒 Smart Basket — Complete YOLO Training Notebook

**المشروع:** Smart Shopping Cart — Raspberry Pi + YOLO + Firebase + Flutter

---

## قبل ما تبدأ:

1. **غير الـ Runtime لـ GPU:**
   - Runtime → Change runtime type → GPU (T4)
   - دي خطوة مهمة جداً — التدريب على CPU بياخد 10x وقت أكتر

2. **تأكد إن الـ GPU شغال:**
   - شغل الـ Cell الأول (Environment Check)
   - لازم يظهر `CUDA available: True`

3. **عندك الـ Dataset على Roboflow:**
   - محتاج الـ API Key وwebspace وproject name
   - هتحطهم في Cell رقم 3

---

## الـ Cells بالترتيب:

| Cell | الوظيفة | الوقت التقريبي |
|------|---------|---------------|
| 1 | Environment Check | ثوانٍ |
| 2 | Install Dependencies | 2-3 دقائق |
| 3 | Download Dataset | 1-5 دقائق |
| 4 | Inspect Dataset | ثوانٍ |
| 5 | Train YOLO26n | 45-90 دقيقة |
| 6 | Validate | 2-3 دقائق |
| 7 | Export ONNX | 1-2 دقيقة |
| 8 | Export NCNN | 2-3 دقائق |
| 9 | Download Results | ثوانٍ |

## Cell 1 — Environment Check
تحقق إن الـ GPU شغال قبل أي حاجة

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: ENVIRONMENT CHECK
# تحقق من الـ hardware والـ software قبل التدريب
# ═══════════════════════════════════════════════════════════════

import torch
import platform
import subprocess

print('=' * 60)
print('  Smart Basket — Environment Check')
print('=' * 60)

# ── Python Version ─────────────────────────────────────────────
print(f'\n  Python version : {platform.python_version()}')

# ── PyTorch ────────────────────────────────────────────────────
print(f'  PyTorch version: {torch.__version__}')

# ── CUDA (GPU) ─────────────────────────────────────────────────
# CUDA هو الـ API بتاع NVIDIA اللي بيسمح لـ PyTorch يشتغل على GPU
# لو مش موجود → الموديل هيتدرب على CPU (10x أبطأ)
cuda_available = torch.cuda.is_available()
print(f'  CUDA available : {cuda_available}')

if cuda_available:
    gpu_name  = torch.cuda.get_device_name(0)
    gpu_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_ver  = torch.version.cuda
    print(f'  GPU name       : {gpu_name}')
    print(f'  GPU memory     : {gpu_mem:.1f} GB')
    print(f'  CUDA version   : {cuda_ver}')
    
    # على Colab T4: 15GB VRAM → batch=32 بدون مشكلة
    if gpu_mem >= 14:
        print('  ✅ Excellent GPU! Recommended batch=32')
    elif gpu_mem >= 8:
        print('  ✅ Good GPU. Recommended batch=16')
    else:
        print('  ⚠️  Limited VRAM. Use batch=8')
else:
    print('  ❌ No GPU found!')
    print('  Action: Runtime → Change runtime type → GPU → T4')

# ── Disk Space ─────────────────────────────────────────────────
# الـ dataset والـ training outputs بياخدوا مساحة
# Colab يوفر حوالي 78GB
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
disk_info = result.stdout.split('\n')[1].split()
print(f'\n  Disk total     : {disk_info[1]}')
print(f'  Disk available : {disk_info[3]}')

# ── RAM ────────────────────────────────────────────────────────
import psutil
ram = psutil.virtual_memory()
print(f'  RAM available  : {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB')

print('\n' + '=' * 60)
print('  Environment check complete ✅')
print('=' * 60)

## Cell 2 — Install Dependencies
تثبيت كل المكتبات المطلوبة

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: INSTALL DEPENDENCIES
# الـ ! في البداية معناه: شغّل الأمر ده في الـ terminal
# مش Python — ده shell command
# ═══════════════════════════════════════════════════════════════

# ── Ultralytics ────────────────────────────────────────────────
# الـ framework الأساسي اللي بيشيل YOLO
# -q = quiet mode (مش بيطبع كل حاجة)
!pip install ultralytics>=8.4.0 -q

# ── Roboflow ───────────────────────────────────────────────────
# عشان نحمل الـ dataset من Roboflow مباشرة في الكود
!pip install roboflow -q

# ── ONNX Export Tools ──────────────────────────────────────────
!pip install onnx onnxruntime onnxslim onnxruntime-tools -q

# ── Augmentation ───────────────────────────────────────────────
# Ultralytics بيستخدم albumentations تلقائياً لو مثبتة
!pip install albumentations -q

# ── Restart message ────────────────────────────────────────────
print('\n✅ All dependencies installed!')
print('⚠️  If you see version warnings, that is OK.')
print('   Continue to the next cell.')

## Cell 3 — Download Dataset from Roboflow
تحميل الـ dataset مباشرة في Colab

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: DOWNLOAD DATASET FROM ROBOFLOW
# غير الـ 3 variables دول بتاعتك
# ═══════════════════════════════════════════════════════════════

from roboflow import Roboflow

# ── تعديل مطلوب منك ───────────────────────────────────────────
# API_KEY: موجود في Roboflow → Settings → API Keys
# لا تعمل commit للـ notebook وفيه الـ API key!
# الأفضل: اخزنه في Colab Secrets
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # ← غير ده
WORKSPACE_NAME   = "smart-basket-workspace"  # ← غير ده (اسم الـ workspace في Roboflow)
PROJECT_NAME     = "grocery-items-v1"  # ← غير ده (اسم الـ project)
DATASET_VERSION  = 1  # ← رقم الـ version اللي عملته

# ── أفضل طريقة: Colab Secrets ─────────────────────────────────
# بدل ما تكتب الـ API key هنا:
# 1. اضغط على 🔑 (Secrets) في الـ sidebar
# 2. أضف Secret اسمه ROBOFLOW_API_KEY وقيمته الـ key
# 3. فك الـ comment من السطر التالي:
# from google.colab import userdata
# ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

print(f'Connecting to Roboflow...')
print(f'  Workspace : {WORKSPACE_NAME}')
print(f'  Project   : {PROJECT_NAME}')
print(f'  Version   : {DATASET_VERSION}')

# ── تحميل الـ Dataset ─────────────────────────────────────────
rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
version = project.version(DATASET_VERSION)

# download بيحمل الصور والـ labels بـ format YOLO
# location = المجلد اللي هيتحمل فيه
dataset = version.download(
    model_format = 'yolov8',  # format الـ labels
    location     = '/content/grocery-items',
    overwrite    = True
)

# مسار الـ data.yaml اللي Roboflow عمله
DATASET_YAML = dataset.location + '/data.yaml'

print(f'\n✅ Dataset downloaded!')
print(f'   Location  : {dataset.location}')
print(f'   data.yaml : {DATASET_YAML}')

## Cell 4 — Inspect Dataset
تفتيش الـ dataset قبل التدريب

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: INSPECT DATASET
# لازم تعرف شكل الـ dataset قبل التدريب
# ═══════════════════════════════════════════════════════════════

import yaml
import os
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import numpy as np
import random

# ── قراءة data.yaml ───────────────────────────────────────────
with open(DATASET_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print('=' * 60)
print('  Dataset Configuration')
print('=' * 60)

nc    = data_cfg.get('nc', 0)
names = data_cfg.get('names', {})

print(f'  Classes: {nc}')
if isinstance(names, dict):
    for i, name in names.items():
        print(f'    {i}: {name}')
else:
    for i, name in enumerate(names):
        print(f'    {i}: {name}')

# ── إحصائيات الصور ────────────────────────────────────────────
dataset_root = Path(dataset.location)

print('\n  Image Counts:')
total_images = 0
split_counts = {}
for split in ['train', 'valid', 'test']:
    img_dir = dataset_root / split / 'images'
    if img_dir.exists():
        imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        count = len(imgs)
        split_counts[split] = count
        total_images += count
        print(f'    {split:<8}: {count} images')
    else:
        print(f'    {split:<8}: NOT FOUND')

print(f'    {"TOTAL":<8}: {total_images} images')

# ── Class Distribution ─────────────────────────────────────────
# كام instance من كل class في الـ training set
print('\n  Counting class instances in training set...')

train_labels_dir = dataset_root / 'train' / 'labels'
class_counts = Counter()

for label_file in train_labels_dir.glob('*.txt'):
    with open(label_file) as f:
        for line in f:
            line = line.strip()
            if line:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

print('\n  Training Set Class Distribution:')
print(f'  {"Class":<25} {"Instances":>10} {"Bar"}')
print('  ' + '-' * 55)

max_count = max(class_counts.values()) if class_counts else 1
name_dict = names if isinstance(names, dict) else {i:n for i,n in enumerate(names)}

for cls_id in sorted(class_counts.keys()):
    cls_name = name_dict.get(cls_id, f'class_{cls_id}')
    count    = class_counts[cls_id]
    bar_len  = int(count / max_count * 30)
    bar      = '█' * bar_len
    print(f'  {cls_name:<25} {count:>10}  {bar}')

# ── Imbalance Check ────────────────────────────────────────────
if class_counts:
    min_count = min(class_counts.values())
    max_count = max(class_counts.values())
    ratio = max_count / min_count
    print(f'\n  Imbalance ratio: {ratio:.2f}')
    if ratio > 2.0:
        print('  ⚠️  WARNING: Imbalance > 2x detected!')
        print('     Consider adding more images for smaller classes')
    else:
        print('  ✅ Class balance is acceptable')

# ── Sample Images ──────────────────────────────────────────────
print('\n  Displaying sample training images...')

train_imgs_dir = dataset_root / 'train' / 'images'
sample_imgs    = random.sample(list(train_imgs_dir.glob('*.jpg')), min(9, len(list(train_imgs_dir.glob('*.jpg')))))

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Sample Training Images', fontsize=14)

for i, ax in enumerate(axes.flatten()):
    if i < len(sample_imgs):
        img  = cv2.imread(str(sample_imgs[i]))
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # قراءة الـ labels
        label_file = train_labels_dir / (sample_imgs[i].stem + '.txt')
        if label_file.exists():
            with open(label_file) as f:
                for line in f:
                    parts  = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = int((cx - bw/2) * w)
                        y1 = int((cy - bh/2) * h)
                        bw_px = int(bw * w)
                        bh_px = int(bh * h)
                        rect = patches.Rectangle((x1, y1), bw_px, bh_px,
                                                  linewidth=2, edgecolor='lime',
                                                  facecolor='none')
                        ax.add_patch(rect)
                        cls_name = name_dict.get(cls_id, str(cls_id))
                        ax.text(x1, y1-5, cls_name[:10], color='lime',
                                fontsize=7, backgroundcolor='black')

        ax.imshow(img)
        ax.set_title(sample_imgs[i].name[:20], fontsize=8)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print('  ✅ Sample images displayed')

## Cell 5 — Train YOLO26n ⭐
**الخطوة الأهم — التدريب الفعلي**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: TRAIN YOLO26n
# ⭐ الأهم في كل الـ notebook
# الـ training بياخد 45-90 دقيقة على T4 GPU
# ═══════════════════════════════════════════════════════════════

from ultralytics import YOLO
import time

# ── اختيار الموديل ────────────────────────────────────────────
# YOLO26n: Nano — 2.5M parameters — للـ Pi 5
# YOLO26s: Small — 11M parameters — لو عايز دقة أعلى وعندك accelerator
# YOLO26m: Medium — 20M parameters — للـ server
#
# القرار: yolo26n للـ graduation project
# السبب:
#   - mAP@50 = ~97% على 10 classes كافي للـ demo
#   - يشتغل بـ ~12 FPS على Pi 5 مع INT8
#   - خفيف على الـ RAM (2.5MB ONNX INT8)
MODEL_NAME = 'yolo26n.pt'  # هيتحمل tلقائياً من الإنترنت لو مش موجود

print('=' * 60)
print(f'  Model     : {MODEL_NAME}')
print(f'  Dataset   : {DATASET_YAML}')
print('=' * 60)

# ── تحميل الموديل ─────────────────────────────────────────────
# YOLO بيحمل الـ pretrained weights من COCO
# COCO: 80 class من الحياة الواقعية (شخص، سيارة، قطة...)
# الـ transfer learning: بياخد الـ features اللي اتعلمها على COCO
# ويستخدمها كـ starting point لـ 10 classes بتاعتنا
# النتيجة: وصول لـ 95%+ mAP بـ 500 صورة لكل class
# بدل training from scratch: يحتاج 10,000+ صورة لكل class!
model = YOLO(MODEL_NAME)

t_start = time.time()

# ── التدريب ───────────────────────────────────────────────────
results = model.train(

    # ─── DATASET ─────────────────────────────────────────
    data   = DATASET_YAML,

    # imgsz = 640
    # الصور بتتـ resize لـ 640×640 قبل التدريب
    # 640 = الـ sweet spot بين accuracy وسرعة
    # أكبر (1280) = أدق للأشياء الصغيرة، بطيء أكتر
    # أصغر (320)  = أسرع، أقل دقة
    imgsz  = 640,

    # batch = -1
    # الـ YOLO يحسب تلقائياً أكبر batch يناسب الـ VRAM
    # على T4 (15GB): هيختار batch=32 أو 64
    # batch أكبر = gradient أكثر استقراراً = نتائج أفضل
    batch  = -1,

    # ─── TRAINING SCHEDULE ───────────────────────────────
    # epochs = 100
    # بس مع patience=20، التدريب ممكن يوقف قبل كده
    epochs   = 100,

    # patience = 20
    # لو الـ mAP ما تحسنش لـ 20 epochs → وقف التدريب
    # Early Stopping: بيحمي من الـ overfitting ويوفر وقت
    patience = 20,

    # ─── OPTIMIZER ───────────────────────────────────────
    # MuSGD: الـ optimizer الخاص بـ YOLO26
    # أفضل من Adam لهذه الـ architecture
    optimizer    = 'MuSGD',

    # lr0 = 0.001: Initial Learning Rate
    # الـ learning rate بيبدأ بقيمة صغيرة عشان ما يكسرش
    # الـ pretrained weights
    lr0          = 0.001,

    # lrf = 0.01: Final LR multiplier
    # في آخر التدريب: lr = 0.001 × 0.01 = 0.00001
    # تقليل تدريجي عشان الموديل يـ converge بدقة
    lrf          = 0.01,

    # ─── AUGMENTATION — GEOMETRIC ────────────────────────
    fliplr       = 0.5,   # 50% mirror horizontally
    flipud       = 0.0,   # لا قلب عمودي
    degrees      = 10.0,  # ±10° rotation
    scale        = 0.5,   # ±50% zoom
    translate    = 0.1,   # ±10% shift
    shear        = 5.0,   # ±5° perspective shear

    # ─── AUGMENTATION — COLOR ────────────────────────────
    # hsv_h صغير جداً — Pepsi أزرق وCoca-Cola أحمر!
    hsv_h        = 0.015,  # ±1.5% hue (KEEP SMALL)
    hsv_s        = 0.5,    # ±50% saturation
    hsv_v        = 0.4,    # ±40% brightness (IMPORTANT)

    # ─── AUGMENTATION — ADVANCED ─────────────────────────
    mosaic       = 1.0,   # دايماً يدمج 4 صور
    mixup        = 0.1,   # 10% image blending
    copy_paste   = 0.1,   # 10% object transplant
    erasing      = 0.4,   # 40% random erase (occlusion training)

    # بيوقف الـ mosaic في آخر 10 epochs عشان يـ converge
    close_mosaic = 10,

    # ─── HARDWARE ────────────────────────────────────────
    device   = 0,      # GPU 0 (الـ T4 على Colab)
    workers  = 4,      # عدد الـ CPU threads لتحميل الصور

    # amp = True: Automatic Mixed Precision
    # بيستخدم float16 بدل float32 حيثما ممكن
    # نتيجة: توفير 50% VRAM + 30% سرعة أعلى
    amp      = True,

    # cache = 'ram': حفظ الصور في RAM عشان التحميل أسرع
    # Colab عنده 12GB RAM — كافي للـ dataset بتاعنا
    cache    = 'ram',

    # ─── OUTPUT ──────────────────────────────────────────
    project  = 'runs/detect',
    name     = 'smart_basket_v1',
    plots    = True,
    verbose  = True,
    seed     = 42,

    # ─── LOSS WEIGHTS ────────────────────────────────────
    # box: وزن الـ bounding box accuracy في الـ loss
    # cls: وزن الـ classification accuracy
    # dfl: Distribution Focal Loss (يحسن دقة حواف الـ box)
    box      = 7.5,
    cls      = 0.5,
    dfl      = 1.5,

    # ─── WARMUP ──────────────────────────────────────────
    # في أول 3 epochs: الـ lr بيبدأ صغير ويكبر تدريجياً
    # يمنع الـ gradient explosion في بداية التدريب
    warmup_epochs   = 3.0,
    warmup_momentum = 0.8,
    warmup_bias_lr  = 0.1,
)

t_end      = time.time()
duration_h = (t_end - t_start) / 3600

# ── النتائج ───────────────────────────────────────────────────
print(f'\n⏱️  Training time: {duration_h:.2f} hours')

# حفظ مسار الموديل للـ cells الجاية
BEST_PT      = f'{results.save_dir}/weights/best.pt'
WEIGHTS_DIR  = f'{results.save_dir}/weights'
SAVE_DIR     = str(results.save_dir)

print(f'\n📊 Training Results:')
metrics = results.results_dict
print(f'   mAP@50    : {metrics.get("metrics/mAP50(B)",    0):.4f}')
print(f'   mAP@50-95 : {metrics.get("metrics/mAP50-95(B)", 0):.4f}')
print(f'   Precision : {metrics.get("metrics/precision(B)", 0):.4f}')
print(f'   Recall    : {metrics.get("metrics/recall(B)",    0):.4f}')
print(f'\n📁 Results dir: {SAVE_DIR}')
print(f'✅ Best model : {BEST_PT}')

## Cell 6 — Validate Best Model
تقييم الموديل على الـ validation set

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: VALIDATE BEST MODEL
# بنقيّم الموديل على صور ما شافهاش أثناء التدريب
# ═══════════════════════════════════════════════════════════════

from ultralytics import YOLO

print('🔍 Loading best model for validation...')
model_best = YOLO(BEST_PT)

# ── Validation على الـ val set ────────────────────────────────
# conf=0.001: نستخدم threshold منخفض جداً
# لأن الـ mAP بيتحسب على كل الـ thresholds
# لو رفعنا الـ conf هنا → هنخسر detections → mAP أقل زوراً
metrics = model_best.val(
    data    = DATASET_YAML,
    imgsz   = 640,
    batch   = 32,
    conf    = 0.001,  # منخفض للـ mAP calculation
    iou     = 0.6,
    split   = 'val',
    device  = 0,
    plots   = True,
    verbose = True,
)

# ── النتائج ───────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  VALIDATION RESULTS')
print('=' * 60)

print(f'\n  Overall:')
print(f'    mAP@50       : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'    mAP@50-95    : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'    Precision    : {metrics.box.mp:.4f}')
print(f'    Recall       : {metrics.box.mr:.4f}')

print(f'\n  Per-class mAP@50:')
class_names = metrics.names
for i, ap50 in enumerate(metrics.box.ap50):
    name  = class_names.get(i, f'class_{i}')
    grade = '🟢' if ap50 >= 0.95 else '🟡' if ap50 >= 0.85 else '🔴'
    print(f'    {grade} {name:<20} : {ap50:.4f}')

print(f'\n  Speed:')
print(f'    Preprocess   : {metrics.speed["preprocess"]:.1f} ms')
print(f'    Inference    : {metrics.speed["inference"]:.1f} ms')
print(f'    Postprocess  : {metrics.speed["postprocess"]:.1f} ms')

## Cell 7 — Export ONNX
تحويل الموديل لـ ONNX للتشغيل على أي device

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: EXPORT TO ONNX (FP32 + INT8)
# ═══════════════════════════════════════════════════════════════

from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

model_best = YOLO(BEST_PT)

# ── ONNX FP32 Export ──────────────────────────────────────────
print('📦 Exporting to ONNX FP32...')
print(f'   imgsz : 416×416 (export size, different from training 640)')
print(f'   ملاحظة: نصغر الحجم من 640 لـ 416 عشان يكون أسرع على الـ Pi')
print(f'           مع خسارة بسيطة جداً في الدقة')

onnx_path = model_best.export(
    format   = 'onnx',
    imgsz    = 416,   # export size — للـ Pi
    opset    = 17,    # ONNX opset version
    simplify = True,  # optimize الـ graph
    half     = False, # FP32 (Pi CPU مش بيدعم FP16)
    dynamic  = False, # fixed input size
)

ONNX_FP32 = WEIGHTS_DIR + '/best.onnx'
fp32_mb   = os.path.getsize(ONNX_FP32) / 1e6
print(f'✅ ONNX FP32 : {ONNX_FP32} ({fp32_mb:.1f} MB)')

# ── ONNX INT8 Quantization ────────────────────────────────────
print('\n📦 Quantizing to INT8...')
print('   FP32: كل وزن = 4 bytes')
print('   INT8: كل وزن = 1 byte')
print('   النتيجة: 4x أصغر، 2-3x أسرع على CPU')

ONNX_INT8 = WEIGHTS_DIR + '/best_int8.onnx'
quantize_dynamic(
    model_input  = ONNX_FP32,
    model_output = ONNX_INT8,
    weight_type  = QuantType.QUInt8,  # unsigned 8-bit integer
)

int8_mb = os.path.getsize(ONNX_INT8) / 1e6
print(f'✅ ONNX INT8 : {ONNX_INT8} ({int8_mb:.1f} MB)')
print(f'   Compression: {fp32_mb/int8_mb:.1f}x smaller')

## Cell 8 — Export NCNN
تحويل الموديل لـ NCNN للـ Raspberry Pi (أسرع format)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: EXPORT TO NCNN
# NCNN هو الأسرع على ARM processors (Raspberry Pi)
# 30-50% أسرع من ONNX Runtime على نفس الـ hardware
# ═══════════════════════════════════════════════════════════════

from ultralytics import YOLO

print('📦 Exporting to NCNN (ARM-optimized format)...')
print('   NCNN = Tencent Neural Network, مصمم للـ mobile والـ embedded')
print('   بيستخدم ARM NEON SIMD instructions تلقائياً')

model_best = YOLO(BEST_PT)

ncnn_path = model_best.export(
    format = 'ncnn',
    imgsz  = 416,
    half   = False,  # Pi 5 مش بيدعم FP16 نيتيفلي
)

import os
print(f'\n✅ NCNN exported to: {ncnn_path}')
print(f'   الملفات:')

from pathlib import Path
ncnn_dir = Path(ncnn_path)
for f in ncnn_dir.iterdir():
    size_mb = f.stat().st_size / 1e6
    print(f'     {f.name}: {size_mb:.1f} MB')

print()
print('   model.param = بنية الموديل (architecture definition)')
print('   model.bin   = الأوزان (trained weights)')
print('   الاتنين مع بعض = الموديل الكامل')

NCNN_DIR = str(ncnn_dir)

## Cell 9 — Create Packages & Download
تحزيم الملفات وتحميلها

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: PACKAGE AND DOWNLOAD
# بنحزم كل حاجة في zip files ونحملها
# ═══════════════════════════════════════════════════════════════

import zipfile
import shutil
from pathlib import Path
from google.colab import files

# ── classes.txt ───────────────────────────────────────────────
classes_txt = WEIGHTS_DIR + '/classes.txt'
with open(classes_txt, 'w') as f:
    class_names = [
        'water_bottle', 'pepsi_can', 'coca_cola_can', 'juice_box',
        'milk_carton', 'chocolate_bar', 'chips_bag', 'biscuits_pack',
        'rice_bag', 'sugar_bag'
    ]
    f.write('\n'.join(class_names))
print('✅ classes.txt created')

# ── Laptop Package ────────────────────────────────────────────
laptop_zip = '/content/laptop_package.zip'
with zipfile.ZipFile(laptop_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_FP32,   'best.onnx')
    zf.write(ONNX_INT8,   'best_int8.onnx')
    zf.write(classes_txt, 'classes.txt')

lz = os.path.getsize(laptop_zip) / 1e6
print(f'✅ laptop_package.zip: {lz:.1f} MB')

# ── Pi Package ────────────────────────────────────────────────
rpi5_zip = '/content/rpi5_package.zip'
with zipfile.ZipFile(rpi5_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_INT8,   'best_int8.onnx')
    # NCNN files
    for ncnn_f in Path(NCNN_DIR).iterdir():
        zf.write(str(ncnn_f), f'ncnn/{ncnn_f.name}')
    zf.write(classes_txt, 'classes.txt')

rz = os.path.getsize(rpi5_zip) / 1e6
print(f'✅ rpi5_package.zip: {rz:.1f} MB')

# ── Training Results Archive ──────────────────────────────────
results_zip = '/content/training_results.zip'
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path(SAVE_DIR).rglob('*'):
        if f.is_file():
            zf.write(str(f), str(f.relative_to(SAVE_DIR)))

rz2 = os.path.getsize(results_zip) / 1e6
print(f'✅ training_results.zip: {rz2:.1f} MB')

# ── Summary ───────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  EXPORT COMPLETE')
print('=' * 60)
print(f'  best.pt           : PyTorch original')
print(f'  best.onnx         : ONNX FP32 ({os.path.getsize(ONNX_FP32)/1e6:.1f} MB)')
print(f'  best_int8.onnx    : ONNX INT8 ({os.path.getsize(ONNX_INT8)/1e6:.1f} MB) ← Pi recommended')
print(f'  ncnn/             : NCNN ARM optimized')
print(f'\n  Packages:')
print(f'  laptop_package.zip : for testing on laptop')
print(f'  rpi5_package.zip   : for Raspberry Pi 5')

# ── Download ──────────────────────────────────────────────────
print('\nDownloading files...')
files.download(laptop_zip)
files.download(rpi5_zip)
# فك الـ comment للـ training results (كبير الحجم)
# files.download(results_zip)
print('\n✅ Downloads started!')

## Cell 10 — Quick Inference Test
اختبار سريع إن الموديل شغال صح

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: QUICK INFERENCE TEST
# تأكد إن الموديل بيكتشف صح على صور من الـ test set
# ═══════════════════════════════════════════════════════════════

from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import random
from pathlib import Path

# اختبار على الـ ONNX INT8 (اللي هنستخدمه على Pi)
print('Testing ONNX INT8 model...')
model_onnx = YOLO(ONNX_INT8)

# اختار 6 صور من الـ test set عشوائياً
test_imgs_dir = Path(dataset.location) / 'test' / 'images'
if not test_imgs_dir.exists():
    test_imgs_dir = Path(dataset.location) / 'valid' / 'images'

test_imgs = list(test_imgs_dir.glob('*.jpg'))
samples   = random.sample(test_imgs, min(6, len(test_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('ONNX INT8 Predictions on Test Set', fontsize=14)

for i, (ax, img_path) in enumerate(zip(axes.flatten(), samples)):
    # run inference
    results = model_onnx.predict(
        source  = str(img_path),
        conf    = 0.45,
        verbose = False,
    )

    # رسم النتيجة
    result_img = results[0].plot()  # BGR with boxes drawn
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    ax.imshow(result_img)
    
    # عنوان الصورة
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        det_names = [model_onnx.names[int(c)] for c in boxes.cls]
        title = ', '.join(set(det_names))
    else:
        title = 'No detections'
    
    ax.set_title(f'{img_path.name[:20]}\n{title}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/test_predictions.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Inference test complete!')
print('   لو شايف bounding boxes صح → الموديل شغال تمام')
print('   لو مش عارف يكتشف → راجع الـ conf threshold أو الـ dataset')